Integration of meteorological data, environmental data, and fire point data

In [5]:
import pandas as pd
import numpy as np
from scipy.spatial import KDTree

# --------------------------
# 第一步：读取并预处理所有基础数据
# --------------------------
# 1. 温度+风速数据
try:
    df_meteo = pd.read_csv("/Users/hexiaolu/Desktop/ERA5_data/ERA5_Land_heilongjiang_2010-2019_combination.csv", encoding="utf-8")
except UnicodeDecodeError:
    df_meteo = pd.read_csv("/Users/hexiaolu/Desktop/ERA5_data/ERA5_Land_heilongjiang_2010-2019_combination.csv", encoding="gbk")

df_meteo = df_meteo[["date", "latitude", "longitude", "t2m_celsius", "wind_speed"]]
df_meteo["date"] = pd.to_datetime(df_meteo["date"], format="%Y-%m-%d", errors="coerce").dt.date
df_meteo = df_meteo[df_meteo["date"].notna()].copy()
df_meteo["latitude"] = df_meteo["latitude"].round(1)
df_meteo["longitude"] = df_meteo["longitude"].round(1)

# 2. 850hPa温度数据
try:
    df_t850 = pd.read_csv("/Users/hexiaolu/Desktop/ERA5_data/850hpa温度_2010-2019_摄氏度.csv", encoding="utf-8")
except UnicodeDecodeError:
    df_t850 = pd.read_csv("/Users/hexiaolu/Desktop/ERA5_data/850hpa温度_2010-2019_摄氏度.csv", encoding="gbk")

df_t850["date"] = pd.to_datetime(df_t850["date"], format="%Y-%m-%d", errors="coerce").dt.date
df_t850 = df_t850[df_t850["date"].notna()].copy()
df_t850["latitude"] = df_t850["latitude"].round(1)
df_t850["longitude"] = df_t850["longitude"].round(1)

# 3. 相对湿度数据
try:
    df_rh = pd.read_csv("/Users/hexiaolu/Desktop/ERA5_data/2010-2019_HLJ_2m_relative-humidity_combination.csv", encoding="utf-8")
except UnicodeDecodeError:
    df_rh = pd.read_csv("/Users/hexiaolu/Desktop/ERA5_data/2010-2019_HLJ_2m_relative-humidity_combination.csv", encoding="gbk")

df_rh = df_rh.rename(columns={"time": "date"})
df_rh["date"] = pd.to_datetime(df_rh["date"], format="%Y-%m-%d", errors="coerce").dt.date
df_rh = df_rh[df_rh["date"].notna()].copy()
df_rh = df_rh[["date", "latitude", "longitude", "2m_relative_humidity"]]
df_rh["latitude"] = df_rh["latitude"].round(1)
df_rh["longitude"] = df_rh["longitude"].round(1)

# 4. 火灾数据
df_fire = pd.read_excel("/Users/hexiaolu/Desktop/CNGF5020/火灾分类_总览.xlsx")
df_fire = df_fire.rename(columns={"acq_date": "date"})
df_fire["date"] = pd.to_datetime(df_fire["date"], format="%Y-%m-%d", errors="coerce").dt.date
df_fire = df_fire[df_fire["date"].notna()].copy()
df_fire = df_fire[["date", "latitude", "longitude", "frp", "火灾类型"]]
df_fire["latitude"] = df_fire["latitude"].round(1)
df_fire["longitude"] = df_fire["longitude"].round(1)
df_fire["is_straw_burn"] = df_fire["火灾类型"].str.contains("秸秆|混合", na=False)  # 修正正则表达式

# 汇总火灾数据
df_fire_agg = df_fire.groupby(["date", "latitude", "longitude"]).agg(
    秸秆火点数量=("is_straw_burn", "sum"),
    平均火点强度=("frp", "mean")
).reset_index()


# --------------------------
# 第二步：合并气象数据
# --------------------------
df_meteo_merged = pd.merge(df_meteo, df_t850, on=["date", "latitude", "longitude"], how="inner")
df_meteo_full = pd.merge(df_meteo_merged, df_rh, on=["date", "latitude", "longitude"], how="inner")
print(f"气象数据合并后形状：{df_meteo_full.shape}")


# --------------------------
# 第三步：匹配火点与气象数据
# --------------------------
# 1. 过滤气象范围内的火点
meteo_lat_min, meteo_lat_max = df_meteo_full["latitude"].min(), df_meteo_full["latitude"].max()
meteo_lon_min, meteo_lon_max = df_meteo_full["longitude"].min(), df_meteo_full["longitude"].max()
df_fire_filtered = df_fire_agg[
    (df_fire_agg["latitude"].between(meteo_lat_min, meteo_lat_max)) &
    (df_fire_agg["longitude"].between(meteo_lon_min, meteo_lon_max))
].copy()
original_fire_count = df_fire_filtered["秸秆火点数量"].sum()
print(f"气象范围内原始火点数量：{original_fire_count}")

# 2. 建立气象网格的空间索引
meteo_grids = {}
for date in df_meteo_full["date"].unique():
    meteo_day = df_meteo_full[df_meteo_full["date"] == date].copy()
    coords = meteo_day[["latitude", "longitude"]].values
    meteo_grids[date] = (coords, KDTree(coords), meteo_day)

# 3. 匹配最近的气象网格
matched_fires = []
for idx, fire_row in df_fire_filtered.iterrows():
    fire_date = fire_row["date"]
    if fire_date not in meteo_grids:
        continue
    coords, tree, meteo_day = meteo_grids[fire_date]
    distance, nearest_idx = tree.query([fire_row["latitude"], fire_row["longitude"]])
    if distance > 0.2:  # 距离阈值
        continue
    nearest_meteo = meteo_day.iloc[nearest_idx]
    matched_fires.append({
        "date": fire_date,
        "fire_latitude": fire_row["latitude"],
        "fire_longitude": fire_row["longitude"],
        "秸秆火点数量": fire_row["秸秆火点数量"],
        "平均火点强度": fire_row["平均火点强度"],
        "meteo_latitude": nearest_meteo["latitude"],
        "meteo_longitude": nearest_meteo["longitude"],
        "匹配距离(°)": round(distance, 3),
        "t2m_celsius": nearest_meteo["t2m_celsius"],
        "wind_speed": nearest_meteo["wind_speed"],
        "2m_relative_humidity": nearest_meteo["2m_relative_humidity"],
        "t850": nearest_meteo["t850"]
    })

print(f"火点与气象匹配完成，生成matched_fires，长度：{len(matched_fires)}")


# --------------------------
# 第四步：按年份匹配PM2.5
# --------------------------
def process_year(target_year, df_pm25_all, df_fire_meteo_all, pm25_threshold=0.3):
    """处理单一年份的匹配"""
    df_fire_meteo = df_fire_meteo_all[
        df_fire_meteo_all["date"].apply(lambda x: x.year == target_year)
    ].copy()
    if len(df_fire_meteo) == 0:
        print(f"{target_year}年无火点-气象数据，跳过")
        return pd.DataFrame()
    
    df_pm25 = df_pm25_all[
        df_pm25_all["date"].apply(lambda x: x.year == target_year)
    ].copy()
    if len(df_pm25) == 0:
        print(f"{target_year}年无PM2.5数据，跳过")
        return pd.DataFrame()
    
    pm25_dates = set(df_pm25["date"].unique())
    df_fire_meteo = df_fire_meteo[df_fire_meteo["date"].isin(pm25_dates)].copy()
    if len(df_fire_meteo) == 0:
        print(f"{target_year}年无重叠日期数据，跳过")
        return pd.DataFrame()
    
    daily_results = []
    for date in pm25_dates:
        fire_day = df_fire_meteo[df_fire_meteo["date"] == date]
        if len(fire_day) == 0:
            continue
        
        pm25_day = df_pm25[df_pm25["date"] == date]
        pm25_coords = pm25_day[["pm25_latitude", "pm25_longitude"]].values
        if len(pm25_coords) == 0:
            continue
        
        kd_tree = KDTree(pm25_coords)
        fire_coords = fire_day[["fire_latitude", "fire_longitude"]].values
        distances, indices = kd_tree.query(fire_coords)
        
        valid_mask = distances <= pm25_threshold
        if not np.any(valid_mask):
            continue
        
        valid_fire = fire_day.iloc[valid_mask].reset_index(drop=True)
        valid_pm25 = pm25_day.iloc[indices[valid_mask]].reset_index(drop=True)
        valid_distances = pd.Series(distances[valid_mask].round(3), name="PM2.5匹配距离(°)")
        
        daily_result = pd.concat([
            valid_fire,
            valid_pm25[["pm25_latitude", "pm25_longitude", "pm25_concentration"]],
            valid_distances
        ], axis=1)
        daily_results.append(daily_result)
    
    year_result = pd.concat(daily_results, ignore_index=True) if daily_results else pd.DataFrame()
    print(f"{target_year}年匹配完成，数据量：{year_result.shape[0]}")
    return year_result


# --------------------------
# 主流程：按年份匹配PM2.5
# --------------------------
if __name__ == "__main__":
    # 预处理PM2.5数据
    df_pm25 = pd.read_csv("/Users/hexiaolu/Desktop/CNGF5020/HLJ_2010-2019_PM2.5-combination.csv")
    df_pm25 = df_pm25.rename(columns={
        "lat": "pm25_latitude",
        "lon": "pm25_longitude",
        "PM2.5(µg/m3)": "pm25_concentration"
    })
    df_pm25["date"] = pd.to_datetime(df_pm25["date"]).dt.date
    df_pm25 = df_pm25[df_pm25["year"] != 2017].copy()
    df_pm25[["pm25_latitude", "pm25_longitude"]] = df_pm25[["pm25_latitude", "pm25_longitude"]].round(1)
    df_pm25 = df_pm25[["date", "pm25_latitude", "pm25_longitude", "pm25_concentration"]]
    print(f"PM2.5数据预处理完成，形状：{df_pm25.shape}")
    
    # 火点-气象数据转换为DataFrame
    df_fire_meteo_all = pd.DataFrame(matched_fires)
    df_fire_meteo_all["date"] = pd.to_datetime(df_fire_meteo_all["date"]).dt.date
    print(f"火点-气象数据加载完成，形状：{df_fire_meteo_all.shape}")
    
    # 按年份处理（2010-2019，跳过2017）
    years = [2010, 2011, 2012, 2013, 2014, 2015, 2016, 2018, 2019]
    all_results = []
    
    for year in years:
        year_result = process_year(
            target_year=year,
            df_pm25_all=df_pm25,
            df_fire_meteo_all=df_fire_meteo_all,
            pm25_threshold=0.3
        )
        if not year_result.empty:
            all_results.append(year_result)
            year_result.to_csv(f"火点_气象_PM25_匹配结果_{year}.csv", index=False)
    
    # 合并所有年份结果并计算逆温强度（单位：℃/100m）
    df_final = pd.concat(all_results, ignore_index=True) if all_results else pd.DataFrame()
    if not df_final.empty:
        # 850hPa高度约为1500米，2米温度高度约为2米，高差约为1500米（简化）
        height_diff = 1500  # 单位：米
        df_final["温度差(℃)"] = df_final["t850"] - df_final["t2m_celsius"]  # 温度差（℃）
        # 计算逆温强度：(温度差 / 高差) * 100 → 转换为℃/100米
        df_final["逆温强度(℃/100m)"] = (df_final["温度差(℃)"] / height_diff) * 100
    
    # 统计与保存
    final_count = df_final["秸秆火点数量"].sum() if not df_final.empty else 0
    print(f"\n所有年份匹配完成，总火点保留率：{final_count/original_fire_count:.2%}，最终数据量：{df_final.shape}")
    df_final.to_csv("火点_气象_PM25_三者匹配结果_总-V1.csv", index=False)
    print("最终结果已保存：火点_气象_PM25_三者匹配结果_总-V1.csv")

气象数据合并后形状：(16620252, 7)
气象范围内原始火点数量：33941
火点与气象匹配完成，生成matched_fires，长度：154651
PM2.5数据预处理完成，形状：(29523834, 4)
火点-气象数据加载完成，形状：(154651, 12)
2010年匹配完成，数据量：8777
2011年匹配完成，数据量：15111
2012年匹配完成，数据量：7731
2013年匹配完成，数据量：11222
2014年匹配完成，数据量：23952
2015年匹配完成，数据量：21933
2016年匹配完成，数据量：17651
2018年匹配完成，数据量：10354
2019年匹配完成，数据量：13233

所有年份匹配完成，总火点保留率：90.17%，最终数据量：(129964, 18)
最终结果已保存：火点_气象_PM25_三者匹配结果_总-V1.csv


In [7]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# 加载合并后的数据
df = pd.read_csv("火点_气象_PM25_三者匹配结果_总-V1.csv")
# 转换date为 datetime 类型（便于时间分析）
df["date"] = pd.to_datetime(df["date"])
# 查看基本信息（字段、数据类型、缺失值）
print(df.info())
print(df.describe())  # 数值型变量的统计量（均值、标准差等）

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 129964 entries, 0 to 129963
Data columns (total 18 columns):
 #   Column                Non-Null Count   Dtype         
---  ------                --------------   -----         
 0   date                  129964 non-null  datetime64[ns]
 1   fire_latitude         129964 non-null  float64       
 2   fire_longitude        129964 non-null  float64       
 3   秸秆火点数量                129964 non-null  int64         
 4   平均火点强度                129964 non-null  float64       
 5   meteo_latitude        129964 non-null  float64       
 6   meteo_longitude       129964 non-null  float64       
 7   匹配距离(°)               129964 non-null  float64       
 8   t2m_celsius           129964 non-null  float64       
 9   wind_speed            129964 non-null  float64       
 10  2m_relative_humidity  124508 non-null  float64       
 11  t850                  60859 non-null   float64       
 12  pm25_latitude         129964 non-null  float64       
 13 